# Tenacious-Bench Week 11 — DPO Preference Tuning (Path B)
**Model:** Qwen2.5-1.5B-Instruct (4-bit QLoRA via Unsloth)
**Method:** Direct Preference Optimization (DPO)
**Hardware:** Google Colab T4 GPU

> Run each cell in order and wait for it to fully complete before moving to the next.

## Step 0 — Mount Google Drive (saves your model permanently!)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted!')

## Step 1 — Install Unsloth and Dependencies

In [ ]:
# Verify GPU is available first
!nvidia-smi


In [ ]:
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" unsloth-zoo
!pip install --no-deps xformers trl peft accelerate bitsandbytes
print('Installation complete!')

## Step 2 — Load Qwen2.5-1.5B Model with LoRA Adapters

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048

# Load 1.5B model — fits comfortably in T4 (14.5GB VRAM)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Qwen2.5-1.5B-Instruct',
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# Attach LoRA adapters for efficient fine-tuning
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                      'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state = 3407,
)

free_gb = torch.cuda.mem_get_info()[0] / 1024**3
print(f'Model + LoRA loaded! Free GPU memory: {free_gb:.1f} GB')

## Step 3 — Upload and Load Preference Datasets
> **Before running this cell:** Upload `preferences_train.jsonl` and `preferences_dev.jsonl` using the 📁 folder icon on the left sidebar in Colab.

In [ ]:
import json
from datasets import Dataset, disable_caching

# Disable HuggingFace caching to avoid pickling issues
disable_caching()

def read_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

# Load the preference pair files
train_rows = read_jsonl('/content/preferences_train.jsonl')
dev_rows   = read_jsonl('/content/preferences_dev.jsonl')

# Format as DPO triples (prompt, chosen, rejected)
def format_row(row):
    return {
        'prompt':   f"<|im_start|>user\n{row['prompt']}<|im_end|>\n<|im_start|>assistant\n",
        'chosen':   f"{row['chosen']}<|im_end|>",
        'rejected': f"{row['rejected']}<|im_end|>"
    }

train_dataset = Dataset.from_list([format_row(r) for r in train_rows])
eval_dataset  = Dataset.from_list([format_row(r) for r in dev_rows])

print(f'Loaded — train: {len(train_dataset)} pairs, eval: {len(eval_dataset)} pairs')
print('Sample prompt preview:')
print(train_dataset[0]['prompt'][:150])

## Step 4 — Launch DPO Training (~30 minutes)

In [ ]:
from trl import DPOTrainer, DPOConfig

trainer = DPOTrainer(
    model = model,
    ref_model = None,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    max_length = 1024,
    max_prompt_length = 512,
    args = DPOConfig(
        dataset_num_proc = 0,                          # Avoids pickling crash
        per_device_train_batch_size = 1,               # Low batch to avoid OOM
        gradient_accumulation_steps = 8,               # Effective batch size = 8
        max_steps = 60,                                # ~6 full epochs over 77 pairs
        learning_rate = 2e-5,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = '/content/drive/MyDrive/Tenacious-Qwen-DPO',  # Saved to Drive!
        beta = 0.1,
    ),
)

trainer.train()

## Step 5 — Plot the Convergence Curve

In [ ]:
import matplotlib.pyplot as plt

history = trainer.state.log_history
steps       = [x['step'] for x in history if 'loss' in x]
loss_values = [x['loss'] for x in history if 'loss' in x]

plt.figure(figsize=(10, 5))
plt.plot(steps, loss_values, color='#ff7f0e', linewidth=2, label='DPO Loss')
plt.title('Tenacious-Bench: DPO Preference Tuning Convergence', fontsize=14, fontweight='bold')
plt.xlabel('Training Steps', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

if loss_values:
    plt.annotate(
        f'Final Loss: {loss_values[-1]:.4f}',
        xy=(steps[-1], loss_values[-1]),
        xytext=(steps[-1] - 15, loss_values[-1] + 0.05),
        arrowprops=dict(facecolor='black', shrink=0.05)
    )

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Tenacious-Qwen-DPO/convergence_plot.png', dpi=150)
plt.show()
print('Plot saved to Google Drive!')

## Step 6 — Push LoRA Adapter to Hugging Face Hub

In [ ]:
import os
from huggingface_hub import HfApi

# Paste your HuggingFace WRITE token here (starts with hf_...)
HF_TOKEN = 'hf_YOUR_WRITE_TOKEN_HERE'

api = HfApi()

# Confirm token works
whoami = api.whoami(token=HF_TOKEN)
print(f'Logged in as: {whoami["name"]}')

# Find checkpoint folder
drive_output = '/content/drive/MyDrive/Tenacious-Qwen-DPO'
checkpoint_dirs = [d for d in os.listdir(drive_output) if d.startswith('checkpoint')]
checkpoint_path = os.path.join(drive_output, sorted(checkpoint_dirs)[-1])
print(f'Uploading from: {checkpoint_path}')

# Push to HuggingFace
api.upload_folder(
    folder_path = checkpoint_path,
    repo_id = f"{whoami['name']}/Tenacious-Qwen-DPO",
    repo_type = 'model',
    token = HF_TOKEN
)
print(f"\nSuccess! Model published at: https://huggingface.co/{whoami['name']}/Tenacious-Qwen-DPO")